# Cross-dataset #2 — calibrate PanNuke, test CoNSeP (total count)

**Second real-shift target** (after NuInsSeg). CoNSeP = 41 H&E tiles 1000x1000
(Graham 2019), tiled into 4x4 = 16 patches of 250x250 (~656 patches) so per-patch
count scale matches PanNuke -> a genuine **domain shift**, not a scale artifact.

Same story as Table 6b: split conformal calibrated on PanNuke under-covers on CoNSeP;
ACI / PB-JCI Online (streaming feedback) recover. Confirms the effect generalizes.

**Attach:** consep.zip dataset, `hipinhththu/sam3-native-pt`,
`hipinhththu/sam3-q1-multiseed-ckpts` (LoRA seed42 + `phase_C_preds_seed42.pkl`).

## 00 — Setup

In [ ]:
import subprocess, sys, os, glob, time, json, zipfile
import numpy as np
import torch
from PIL import Image
import scipy.io as sio
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"

LORA_CANDIDATES = [
    "/kaggle/input/datasets/hipinhththu/sam3-q1-multiseed-ckpts/sam3_lora_seed42_final.pt",
    "/kaggle/input/sam3-q1-multiseed-ckpts/sam3_lora_seed42_final.pt",
    "/kaggle/input/datasets/hipinhththu/phase-a2-lora-weights/sam3_lora_rank16_final.pt",
]
LORA_PATH = next((p for p in LORA_CANDIDATES if os.path.exists(p)), None)
if LORA_PATH is None:
    hits = glob.glob("/kaggle/input/**/sam3_lora_seed42_final.pt", recursive=True) \
        or glob.glob("/kaggle/input/**/sam3_lora_rank16_final.pt", recursive=True)
    LORA_PATH = hits[0] if hits else None

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR], check=True)
assert os.path.exists(CHECKPOINT_PATH), "Attach hipinhththu/sam3-native-pt"
assert LORA_PATH, "Attach sam3-q1-multiseed-ckpts (sam3_lora_seed42_final.pt)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scipy", "opencv-python", "einops", "tqdm"], check=True)
print("OK setup | LoRA:", LORA_PATH)

## Helper modules

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile lora_sam3.py
from __future__ import annotations
import math
from typing import Iterable, List, Optional, Set, Tuple
import torch
import torch.nn as nn

class LoRALinear(nn.Module):

    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32,
                 dropout: float = 0.05):
        super().__init__()
        self.base = base
        
        for p in self.base.parameters():
            p.requires_grad = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_f = base.in_features
        out_f = base.out_features
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return out + lora_out * self.scaling

    @property
    def in_features(self) -> int:
        return self.base.in_features

    @property
    def out_features(self) -> int:
        return self.base.out_features

DEFAULT_LORA_TARGETS: Set[str] = {
    
    
    
    "linear1", "linear2",
}

def inject_lora(
    model: nn.Module,
    target_module_names: Iterable[str] = DEFAULT_LORA_TARGETS,
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
    path_must_contain: str = "decoder",
    verbose: bool = True,
) -> Tuple[List[str], int]:
    target_set = set(target_module_names)
    replaced: List[str] = []

    for parent_name, parent in list(model.named_modules()):
        
        if path_must_contain and path_must_contain not in parent_name:
            continue
        for attr_name, child in list(parent.named_children()):
            if attr_name in target_set and isinstance(child, nn.Linear):
                lora_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
                
                base_device = next(child.parameters()).device
                lora_layer.lora_A.data = lora_layer.lora_A.data.to(base_device)
                lora_layer.lora_B.data = lora_layer.lora_B.data.to(base_device)
                setattr(parent, attr_name, lora_layer)
                full_path = f"{parent_name}.{attr_name}" if parent_name else attr_name
                replaced.append(full_path)

    n_lora_params = sum(p.numel() for p in model.parameters()
                        if p.requires_grad)

    if verbose:
        print(f"Injected LoRA into {len(replaced)} modules.")
        print(f"LoRA trainable params: {n_lora_params:,} "
              f"({n_lora_params/1e6:.2f}M)")
        if len(replaced) > 0:
            print("Sample paths:")
            for p in replaced[:5]:
                print(f"  {p}")
            if len(replaced) > 5:
                print(f"  ... ({len(replaced)-5} more)")

    return replaced, n_lora_params

def freeze_non_lora(model: nn.Module) -> Tuple[int, int]:
    n_train = 0
    n_total = 0
    for name, p in model.named_parameters():
        n_total += p.numel()
        if "lora_A" in name or "lora_B" in name:
            p.requires_grad = True
            n_train += p.numel()
        else:
            p.requires_grad = False
    return n_train, n_total

def save_lora_state(model: nn.Module, path: str):
    state = {n: p.detach().cpu()
             for n, p in model.named_parameters()
             if ("lora_A" in n or "lora_B" in n)}
    torch.save(state, path)
    return len(state)

def load_lora_state(model: nn.Module, path: str) -> int:
    state = torch.load(path, map_location="cpu")
    own = dict(model.named_parameters())
    n_loaded = 0
    for k, v in state.items():
        if k in own:
            own[k].data = v.to(own[k].device)
            n_loaded += 1
    return n_loaded

In [ ]:
%%writefile sam3_train.py
from __future__ import annotations
from typing import Dict, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

def make_transform(resolution: int = 1008):
    return v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(resolution, resolution)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

@torch.no_grad()
def encode_image_frozen(model, transform, pil_img, device: str = "cuda"):
    image = v2.functional.to_image(pil_img).to(device)
    image = transform(image).unsqueeze(0)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        backbone_out = model.backbone.forward_image(image)

        
        inst_pred = getattr(model, "inst_interactive_predictor", None)
        if inst_pred is not None and "sam2_backbone_out" in backbone_out:
            sam2_bb = backbone_out["sam2_backbone_out"]
            sam2_bb["backbone_fpn"][0] = (
                inst_pred.model.sam_mask_decoder.conv_s0(sam2_bb["backbone_fpn"][0])
            )
            sam2_bb["backbone_fpn"][1] = (
                inst_pred.model.sam_mask_decoder.conv_s1(sam2_bb["backbone_fpn"][1])
            )
    return backbone_out

def encode_text(model, prompt: str, device: str = "cuda"):
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            return model.backbone.forward_text([prompt], device=device)

def forward_decoder_with_grad(model, backbone_out: Dict, find_stage,
                              geometric_prompt, device: str = "cuda") -> Dict:
    was_training = model.training
    if was_training:
        model.eval()
    try:
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            outputs = model.forward_grounding(
                backbone_out=backbone_out,
                find_input=find_stage,
                geometric_prompt=geometric_prompt,
                find_target=None,
            )
    finally:
        if was_training:
            model.train()
    return outputs

def semantic_union_mask(outputs: Dict, target_hw: Tuple[int, int]) -> torch.Tensor:
    
    pred_logits = outputs["pred_logits"].float()         
    pred_masks  = outputs["pred_masks"].float()          
    pres_logit  = outputs["presence_logit_dec"].float()  

    
    cls_prob = pred_logits.sigmoid()                   
    pres     = pres_logit.sigmoid().unsqueeze(1)       
    prob     = (cls_prob * pres).squeeze(-1)           

    
    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid()  

    
    weighted = prob[:, :, None, None] * masks   

    
    
    
    eps = 1e-4
    log_one_minus = torch.log(torch.clamp(1.0 - weighted, min=eps, max=1.0 - eps))
    log_prod = log_one_minus.sum(dim=1)         
    union = 1.0 - torch.exp(torch.clamp(log_prod, min=-20.0))  
    union = torch.clamp(union, min=eps, max=1.0 - eps)  
    return union.squeeze(0)                     

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def bce_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    p = torch.clamp(pred, eps, 1 - eps)
    return -(target * torch.log(p) + (1 - target) * torch.log(1 - p)).mean()

def semantic_seg_loss(pred: torch.Tensor, target: torch.Tensor,
                      bce_weight: float = 0.5,
                      dice_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    total = bce_weight * bce + dice_weight * dice
    return total, {"bce": float(bce.item()), "dice": float(dice.item()),
                   "loss": float(total.item())}

def compute_loss(pred: torch.Tensor, target: torch.Tensor,
                 bce_weight: float = 0.5, dice_weight: float = 1.0) -> torch.Tensor:
    total, _ = semantic_seg_loss(pred, target, bce_weight, dice_weight)
    return total

@torch.no_grad()
def inference_to_binary(outputs: Dict, target_hw: Tuple[int, int],
                        score_threshold: float = 0.3) -> torch.Tensor:
    pred_logits = outputs["pred_logits"]
    pred_masks  = outputs["pred_masks"]
    pres_logit  = outputs["presence_logit_dec"]

    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)  

    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)  

    keep = prob > score_threshold
    if keep.sum() == 0:
        return torch.zeros(target_hw, dtype=torch.bool, device=pred_logits.device)

    selected = (masks[keep] > 0.5)
    union = selected.any(dim=0)
    return union

In [ ]:
%%writefile conformal.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import numpy as np

def empirical_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return float("inf")
    level = np.ceil((n + 1) * (1 - alpha)) / n
    level = min(level, 1.0)
    return float(np.quantile(scores, level, method="higher"))

def pb_count(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    return (scores[:, None] * probs).sum(axis=0)

def pb_variance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    w = scores[:, None] * probs
    return (w * (1.0 - w)).sum(axis=0)

def pb_covariance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    K = probs.shape[1]
    cov = np.zeros((K, K))
    for j in range(K):
        for k in range(K):
            delta = 1.0 if j == k else 0.0
            cov[j, k] = (scores * probs[:, j] * (delta - probs[:, k])).sum()
    return cov

class MarginalSplitConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "MarginalSplitConformal":
        K = cal_gt[0].shape[0]
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                
                for k in range(K):
                    scores_per_class[k].append(float(gt[k]))
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                err = abs(gt[k] - n_pred[k]) / sigma[k]
                scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]), self.alpha)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

class AdaptiveConformalInference:
    def __init__(self, alpha_target: float = 0.1, gamma: float = 0.05):
        self.alpha_target = alpha_target
        self.gamma = gamma
        self.alpha_t = alpha_target
        self.history_q: List[float] = []
        self.history_scores: List[float] = []  

    def reset(self):
        self.alpha_t = self.alpha_target
        self.history_scores = []

    def update(self, score_t: float, covered_t: bool):
        self.history_scores.append(score_t)
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + self.gamma * (self.alpha_target - err_t)
        
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

    def get_quantile(self) -> float:
        if not self.history_scores:
            return 1.0
        return empirical_quantile(np.array(self.history_scores), self.alpha_t)

class ShiftAwareACI(AdaptiveConformalInference):
    def __init__(self, alpha_target: float = 0.1, gamma_0: float = 0.05,
                 lambda_: float = 3.0, gamma_max: float = 0.15):
        super().__init__(alpha_target, gamma_0)
        self.gamma_0 = gamma_0
        self.lambda_ = lambda_
        self.gamma_max = gamma_max
        self.gamma_t_last = gamma_0

    def update(self, score_t: float, covered_t: bool, delta_t: float = 0.0):
        self.history_scores.append(score_t)
        gamma_t = self.gamma_0 * (1.0 + self.lambda_ * max(0.0, delta_t))
        gamma_t = min(gamma_t, self.gamma_max)
        self.gamma_t_last = gamma_t
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + gamma_t * (self.alpha_target - err_t)
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

class RollingShiftDetector:
    def __init__(self, window: int = 100, cap: float = 1.0):
        self.window = window
        self.cap = cap
        self.baseline: Optional[float] = None
        self.recent: List[float] = []

    def fit_baseline(self, cal_scores) -> "RollingShiftDetector":
        self.baseline = float(np.median(np.asarray(cal_scores))) + 1e-6
        return self

    def step(self, score_t: float) -> float:
        self.recent.append(float(score_t))
        if len(self.recent) > self.window:
            self.recent.pop(0)
        cur = float(np.median(self.recent))
        delta = (cur - self.baseline) / self.baseline
        return float(np.clip(delta, 0.0, self.cap))

class PBAwareJointConformalOnline:
    def __init__(self, alpha: float = 0.1, window: int = 300):
        self.alpha = alpha
        self.window = window
        self.scores: List[float] = []

    def warmstart(self, cal_scores) -> "PBAwareJointConformalOnline":
        self.scores = list(np.asarray(cal_scores)[-self.window:])
        return self

    def get_quantile(self) -> float:
        if not self.scores:
            return float("inf")
        return empirical_quantile(np.asarray(self.scores[-self.window:]), self.alpha)

    def update(self, score_t: float):
        self.scores.append(float(score_t))
        if len(self.scores) > self.window:
            self.scores = self.scores[-self.window:]

def local_coverage_stats(covered_list, window: int = 100) -> Dict[str, float]:
    c = np.asarray(covered_list, dtype=float)
    n = len(c)
    if n == 0:
        return {"min_local_cov": 0.0, "max_miss_run": 0}
    if n >= window:
        roll = np.convolve(c, np.ones(window) / window, mode="valid")
        min_local = float(roll.min())
    else:
        min_local = float(c.mean())
    run = mx = 0
    for v in covered_list:
        run = 0 if v else run + 1
        mx = max(mx, run)
    return {"min_local_cov": min_local, "max_miss_run": int(mx)}

class PBAwareJointConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q: float = 0.0

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "PBAwareJointConformal":
        scores = []
        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            K = len(gt)
            if len(s) == 0:
                
                sigma_eps = 1.0
                S_t = max(abs(gt[k]) / sigma_eps for k in range(K))
            else:
                n_pred = pb_count(s, p)
                sigma = np.sqrt(pb_variance(s, p) + 1e-6)
                S_t = max(abs(gt[k] - n_pred[k]) / sigma[k] for k in range(K))
            scores.append(S_t)
        self.q = empirical_quantile(np.array(scores), self.alpha)
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        s = pred["scores"]
        p = pred["probs"]
        K = pred.get("K", 5)
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q * sigma)
        upper = n_pred + self.q * sigma
        return lower, upper

class ClassStratifiedConformal:
    def __init__(self, alpha: float = 0.1, bonferroni: bool = True):
        self.alpha = alpha
        self.bonferroni = bonferroni
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "ClassStratifiedConformal":
        K = cal_gt[0].shape[0]
        alpha_eff = self.alpha / K if self.bonferroni else self.alpha
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                if gt[k] > 0:  
                    err = abs(gt[k] - n_pred[k]) / sigma[k]
                    scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]) if scores_per_class[k]
                              else np.array([1.0]), alpha_eff)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

def coverage_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                       gt_counts: np.ndarray) -> np.ndarray:
    covered = (gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)
    return covered.mean(axis=0)

def joint_coverage(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                   gt_counts: np.ndarray) -> float:
    covered_all = ((gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)).all(axis=1)
    return float(covered_all.mean())

def avg_width_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> np.ndarray:
    return (intervals_hi - intervals_lo).mean(axis=0)

def macro_width(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> float:
    return float(avg_width_per_class(intervals_lo, intervals_hi).mean())

def split_calibration_test(preds: List[Dict], gts: List[np.ndarray],
                           cal_ratio: float = 0.5,
                           seed: int = 42) -> Tuple[List, List, List, List]:
    n = len(preds)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n)
    n_cal = int(n * cal_ratio)
    cal_idx = indices[:n_cal]
    test_idx = indices[n_cal:]
    cal_preds = [preds[i] for i in cal_idx]
    cal_gt = [gts[i] for i in cal_idx]
    test_preds = [preds[i] for i in test_idx]
    test_gt = [gts[i] for i in test_idx]
    return cal_preds, cal_gt, test_preds, test_gt

In [ ]:
import sys
for p in [".", WORK, SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
from lora_sam3 import inject_lora, load_lora_state, DEFAULT_LORA_TARGETS
from sam3_train import make_transform, encode_image_frozen, encode_text, forward_decoder_with_grad
from conformal import (AdaptiveConformalInference, PBAwareJointConformalOnline,
                       empirical_quantile, pb_count, pb_variance)
print("Helpers loaded.")

## 01 — Locate CoNSeP (handles BOTH zipped and Kaggle-auto-extracted)

Kaggle usually auto-extracts an uploaded `.zip` -> the dataset is already a folder
tree. We first look for an already-extracted `Train/` dir under /kaggle/input; if
none, we find `consep.zip` and extract it to working. Either way we end with `BASE`
= the dir holding `{Train,Test}/{Images,Labels}`.

In [ ]:
def find_base_extracted():
    # any dir under /kaggle/input that has a Train subdir with Images/ in it
    for tr in glob.glob("/kaggle/input/**/Train", recursive=True):
        if os.path.isdir(os.path.join(tr, "Images")):
            return os.path.dirname(tr)
    return None

def find_zip():
    for pat in ["/kaggle/input/**/consep.zip", "/kaggle/input/**/CoNSeP*.zip",
                "/kaggle/input/**/*onsep*.zip"]:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None

BASE = find_base_extracted()
if BASE is None:                          # not auto-extracted -> find + unzip the .zip
    zpath = find_zip()
    assert zpath, "Neither an extracted Train/ dir nor consep.zip found - attach CoNSeP"
    print("zip:", zpath)
    EX = f"{WORK}/consep_data"
    if not os.path.isdir(EX):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(EX)
        print("extracted ->", EX)
    trs = [os.path.dirname(t) for t in glob.glob(f"{EX}/**/Train", recursive=True)
           if os.path.isdir(os.path.join(t, "Images"))]
    assert trs, "No Train/Images found after unzip"
    BASE = trs[0]
print("CoNSeP base:", BASE, "| has", os.listdir(BASE))

## 02 — Index + tile (1000x1000 -> 16x 250x250), GT count per tile

GT count per tile = #instances whose centroid lands in the tile (each instance
assigned to exactly one tile -> tile counts sum to the whole-image count).

In [ ]:
TILE = 250            # 1000 / 250 = 4 -> 4x4 = 16 tiles per image
GRID = 1000 // TILE   # = 4

def list_consep(base):
    items = []
    for split in ["Train", "Test"]:
        idir, ldir = os.path.join(base, split, "Images"), os.path.join(base, split, "Labels")
        if not os.path.isdir(idir):
            continue
        for f in sorted(os.listdir(idir)):
            if f.lower().endswith((".png", ".tif", ".tiff")):
                stem = os.path.splitext(f)[0]
                mat = os.path.join(ldir, stem + ".mat")
                if os.path.exists(mat):
                    items.append({"image": os.path.join(idir, f), "mat": mat,
                                  "split": split, "stem": stem})
    return items

items = list_consep(BASE)
print(f"Indexed {len(items)} CoNSeP tiles (expect 41: 27 Train + 14 Test)")

def tile_counts(mat_path):
    m = sio.loadmat(mat_path)
    inst = m["inst_map"]
    n_total = int(len(np.unique(inst)) - 1)        # exclude background 0
    cent = np.asarray(m["inst_centroid"], dtype=float)  # (N,2) = (x=col, y=row)
    if cent.size == 0:
        return np.zeros(GRID * GRID, dtype=int), n_total
    x, y = cent[:, 0], cent[:, 1]
    col = np.clip((x // TILE).astype(int), 0, GRID - 1)
    row = np.clip((y // TILE).astype(int), 0, GRID - 1)
    tid = row * GRID + col
    counts = np.bincount(tid, minlength=GRID * GRID)
    return counts, n_total

# sanity: tile counts must conserve the whole-image instance count
c0, n0 = tile_counts(items[0]["mat"])
print(f"sample {items[0]['stem']}: whole={n0}  sum(tiles)={c0.sum()}  conserved={c0.sum()==n0}")
assert c0.sum() == n0, "centroid orientation wrong - tiles do not conserve total count"
print("tile counts (4x4):\n", c0.reshape(GRID, GRID))

## 03 — Build SAM3 + A2 LoRA; total-count inference fn

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.data_misc import FindStage
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam3_image_model(device=device, eval_mode=True,
                               checkpoint_path=CHECKPOINT_PATH, load_from_HF=False)
model.eval()
inject_lora(model, target_module_names=DEFAULT_LORA_TARGETS, r=16, alpha=32, dropout=0.0)
load_lora_state(model, LORA_PATH)
for p in model.parameters():
    p.requires_grad = False

transform = make_transform(resolution=1008)
find_stage = FindStage(
    img_ids=torch.tensor([0], device=device, dtype=torch.long),
    text_ids=torch.tensor([0], device=device, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None)
INFER_PROMPT = "cell"
SCORE_THRESH = 0.3
print("SAM3 + A2 LoRA ready.")

@torch.no_grad()
def predict_total(pil_img):
    bb = encode_image_frozen(model, transform, pil_img, device=device)
    bb.update(encode_text(model, INFER_PROMPT, device=device))
    outputs = forward_decoder_with_grad(model, bb, find_stage, model._get_dummy_prompt())
    cls_prob = outputs["pred_logits"].float().sigmoid()
    pres = outputs["presence_logit_dec"].float().sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)
    keep = prob > SCORE_THRESH
    return prob[keep].cpu().numpy() if keep.sum() > 0 else np.zeros(0)

## 04 — Smoke: first image, 16 tiles

In [ ]:
it0 = items[0]
img0 = np.asarray(Image.open(it0["image"]).convert("RGB"))
cnts0, _ = tile_counts(it0["mat"])
for r in range(GRID):
    for c in range(GRID):
        crop = img0[r*TILE:(r+1)*TILE, c*TILE:(c+1)*TILE]
        sc = predict_total(Image.fromarray(crop))
        print(f"  tile({r},{c}) GT={cnts0[r*GRID+c]:3d} | n_det={len(sc):3d} | pred={sc.sum():6.1f}")
print("\nIf pred is in a sane ballpark vs GT -> run full.")

## 05 — Full inference over all tiles (cached) -> consep_preds.pkl

In [ ]:
import pickle
from tqdm import tqdm
CACHE = f"{WORK}/consep_preds.pkl"

if os.path.exists(CACHE):
    with open(CACHE, "rb") as f:
        data = pickle.load(f)
    preds, gts, src = data["preds"], data["gts"], data["src"]
    print(f"Loaded cache: {len(preds)} tiles")
else:
    preds, gts, src = [], [], []
    t0 = time.time()
    for it in tqdm(items, desc="CoNSeP infer"):
        img = np.asarray(Image.open(it["image"]).convert("RGB"))
        cnts, _ = tile_counts(it["mat"])
        for r in range(GRID):
            for c in range(GRID):
                crop = img[r*TILE:(r+1)*TILE, c*TILE:(c+1)*TILE]
                sc = predict_total(Image.fromarray(crop))
                preds.append({"scores": sc, "probs": np.ones((len(sc), 1)), "K": 1})
                gts.append(np.array([float(cnts[r*GRID+c])]))
                src.append(f"{it['stem']}_r{r}c{c}")
    with open(CACHE, "wb") as f:
        pickle.dump({"preds": preds, "gts": gts, "src": src}, f)
    print(f"Done {len(preds)} tiles in {(time.time()-t0)/60:.1f} min -> {CACHE}")

gts_arr = np.array([g[0] for g in gts])
pred_cnt = np.array([p["scores"].sum() for p in preds])
mae = np.abs(gts_arr - pred_cnt).mean()
print(f"GT/tile mean={gts_arr.mean():.1f} (min {gts_arr.min():.0f}, max {gts_arr.max():.0f})")
print(f"Pred/tile mean={pred_cnt.mean():.1f} | Total-count MAE = {mae:.2f}")

## 06 — Total-count nonconformity / interval (K=1)

In [ ]:
ALPHA = 0.1

def nonconf(p, gt):
    if len(p["scores"]) == 0: return float(abs(gt[0]))
    n  = pb_count(p["scores"], p["probs"])[0]
    sg = np.sqrt(pb_variance(p["scores"], p["probs"])[0] + 1e-6)
    return abs(gt[0] - n) / sg

def interval(p, q):
    if len(p["scores"]) == 0: return 0.0, 0.0
    n  = pb_count(p["scores"], p["probs"])[0]
    sg = np.sqrt(pb_variance(p["scores"], p["probs"])[0] + 1e-6)
    return max(0.0, n - q * sg), n + q * sg

def cov_width(P, G, q):
    los = np.array([interval(p, q)[0] for p in P])
    his = np.array([interval(p, q)[1] for p in P])
    g   = np.array([gg[0] for gg in G])
    return float(np.mean((g >= los) & (g <= his))), float(np.mean(his - los))

## 07 — Load PanNuke (source) total-count from phase_C_preds_seed42.pkl

In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

pan_path = find("phase_C_preds_seed42.pkl")
assert pan_path, "phase_C_preds_seed42.pkl not found - attach sam3-q1-multiseed-ckpts"
with open(pan_path, "rb") as f:
    dpan = pickle.load(f)
pan_src = dpan["predictions_by_setting"]["in_dist"]
pan_gtc = np.asarray(dpan["gt_counts"])
pan_preds = [{"scores": np.asarray(p["scores"]),
              "probs": np.ones((len(p["scores"]), 1)), "K": 1} for p in pan_src]
pan_gts = [np.array([float(g.sum())]) for g in pan_gtc]
print(f"PanNuke source: {len(pan_preds)} patches | GT total mean={np.mean([g[0] for g in pan_gts]):.1f}")
print(f"CoNSeP target : {len(preds)} tiles | GT total mean={gts_arr.mean():.1f}")

## 08 — CROSS-DATASET: calibrate PanNuke -> test CoNSeP

Split (no adapt) = honest coverage drop. ACI / PB-JCI Online (window=300, same as
all phases) warm-start on PanNuke then stream CoNSeP with feedback (5 stream seeds).

In [ ]:
pan_scores = np.array([nonconf(pan_preds[i], pan_gts[i]) for i in range(len(pan_preds))])
q_cross = empirical_quantile(pan_scores, ALPHA)
print(f"q (calibrated on PanNuke) = {q_cross:.3f}")

split_cov, split_w = cov_width(preds, gts, q_cross)

def stream(kind, nseeds=5):
    covs, ws = [], []
    for sd in range(nseeds):
        order = np.random.RandomState(sd).permutation(len(preds))
        if kind == "aci":
            m = AdaptiveConformalInference(alpha_target=ALPHA, gamma=0.05)
            m.reset(); m.history_scores = list(pan_scores)
        else:
            m = PBAwareJointConformalOnline(alpha=ALPHA, window=300)
            m.warmstart(pan_scores)
        c, w = [], []
        for i in order:
            q = m.get_quantile(); lo, hi = interval(preds[i], q)
            covered = lo <= gts[i][0] <= hi
            c.append(covered); w.append(hi - lo)
            s = nonconf(preds[i], gts[i])
            m.update(s, covered) if kind == "aci" else m.update(s)
        covs.append(np.mean(c)); ws.append(np.mean(w))
    return float(np.mean(covs)), float(np.std(covs)), float(np.mean(ws)), float(np.std(ws))

aci_c, aci_cs, aci_w, aci_ws = stream("aci")
pbo_c, pbo_cs, pbo_w, pbo_ws = stream("pbo")
print("\nCROSS-DATASET (cal PanNuke -> test CoNSeP):")
print(f"  Split (no adapt) : cov {split_cov*100:.1f}% | width {split_w:.2f}")
print(f"  ACI (stream)     : cov {aci_c*100:.1f}+/-{aci_cs*100:.1f}% | width {aci_w:.2f}+/-{aci_ws:.2f}")
print(f"  PB-JCI Online    : cov {pbo_c*100:.1f}+/-{pbo_cs*100:.1f}% | width {pbo_w:.2f}+/-{pbo_ws:.2f}")

## 09 — In-domain CoNSeP reference (calibrate CoNSeP, 5 seeds)

In [ ]:
def indomain(nseeds=5):
    covs, ws = [], []
    for sd in [42, 100, 200, 300, 400][:nseeds]:
        idx = np.random.RandomState(sd).permutation(len(preds))
        ncal = len(idx) // 2
        cal, test = idx[:ncal], idx[ncal:]
        cs = np.array([nonconf(preds[i], gts[i]) for i in cal])
        q = empirical_quantile(cs, ALPHA)
        c, w = cov_width([preds[i] for i in test], [gts[i] for i in test], q)
        covs.append(c); ws.append(w)
    return float(np.mean(covs)), float(np.std(covs)), float(np.mean(ws)), float(np.std(ws))

id_c, id_cs, id_w, id_ws = indomain()
print(f"In-domain split (cal CoNSeP): cov {id_c*100:.1f}+/-{id_cs*100:.1f}% | width {id_w:.2f}")

## 10 — Summary table + save

In [ ]:
print("=" * 78)
print("CROSS-DATASET #2: PanNuke (cal) -> CoNSeP (test) | total count, alpha=0.1")
print("=" * 78)
print(f"{'Setting / Method':38s} | {'Coverage':>14s} | {'Width':>10s}")
print("-" * 78)
print(f"{'In-domain split (cal CoNSeP)':38s} | {id_c*100:>6.1f}+/-{id_cs*100:<4.1f}% | {id_w:>7.2f}")
print(f"{'Cross split (cal PanNuke, no adapt)':38s} | {split_cov*100:>11.1f}% | {split_w:>7.2f}")
print(f"{'Cross ACI (stream feedback)':38s} | {aci_c*100:>6.1f}+/-{aci_cs*100:<4.1f}% | {aci_w:>7.2f}")
print(f"{'Cross PB-JCI Online (stream)':38s} | {pbo_c*100:>6.1f}+/-{pbo_cs*100:<4.1f}% | {pbo_w:>7.2f}")
print("-" * 78)
drop = (id_c - split_cov) * 100
print(f"\nCoverage DROP (in-domain -> cross split): {drop:+.1f} pp")

out = {
    "dataset": "CoNSeP", "n_tiles": len(preds), "tile_px": TILE, "total_count_MAE": float(mae),
    "in_domain_split":   {"coverage": [id_c, id_cs], "width": [id_w, id_ws]},
    "cross_split":       {"coverage": split_cov, "width": split_w},
    "cross_aci":         {"coverage": [aci_c, aci_cs], "width": [aci_w, aci_ws]},
    "cross_pbjci_online":{"coverage": [pbo_c, pbo_cs], "width": [pbo_w, pbo_ws]},
    "q_cross": float(q_cross), "alpha": ALPHA, "pbjci_window": 300,
}
with open(f"{WORK}/consep_crossdataset_results.json", "w") as f:
    json.dump(out, f, indent=2)
print(f"\nSaved: {WORK}/consep_crossdataset_results.json  (+ consep_preds.pkl)")

## Notes

- **CoNSeP = 2nd independent cross-dataset target** -> Table 6b becomes PanNuke->{NuInsSeg, CoNSeP}.
- Total-count (K=1): no cell-type taxonomy mapping needed (CoNSeP types differ from PanNuke).
- Same window=300, same streaming-feedback assumption as NuInsSeg -> directly comparable.
- Send `consep_crossdataset_results.json` to add the CoNSeP rows to PAPER_TABLES Table 6b.